# pyINLA Interactive Demo

Welcome to **pyINLA** - Fast Bayesian inference for Latent Gaussian Models in Python.

This notebook lets you try pyINLA directly in your browser.

## 1. Setup

First, let's install pyINLA and download the correct binary for Colab.

In [ ]:
# Install pyinla
!pip install pyinla -q
print("pyinla installed!")

In [ ]:
# Download the INLA binary (Ubuntu 22.04 for Colab)
import pyinla

if not pyinla.is_binary_installed():
    print("Downloading INLA binary for Ubuntu 22.04...")
    pyinla.download_binary(os_name="Ubuntu-22.04", interactive=False)
    print("Done!")
else:
    print("INLA binary already installed")

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyinla import pyinla

print("All libraries loaded!")

## 2. Linear Regression: Student Test Scores

Let's predict student test scores based on hours studied.

**Model:** score = intercept + slope × hours + noise

In [ ]:
# Simulate student data
np.random.seed(42)

n = 100
hours_studied = np.random.uniform(1, 10, n)  # Hours studied per week

# True parameters
true_intercept = 40  # Base score with no studying
true_slope = 5       # Points gained per hour of study
noise = np.random.normal(0, 5, n)  # Random variation

# Generate scores
score = true_intercept + true_slope * hours_studied + noise
score = np.clip(score, 0, 100)  # Keep scores in 0-100 range

# Create DataFrame
df = pd.DataFrame({'score': score, 'hours': hours_studied})

print("Sample data:")
print(df.head())
print(f"\nTrue intercept: {true_intercept}")
print(f"True slope: {true_slope}")

In [ ]:
# Visualize the data
plt.figure(figsize=(8, 5))
plt.scatter(df['hours'], df['score'], alpha=0.6, edgecolors='w', linewidth=0.5)
plt.xlabel('Hours Studied per Week')
plt.ylabel('Test Score')
plt.title('Student Test Scores vs Study Time')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Define and fit the model
model = {
    'response': 'score',
    'fixed': ['1', 'hours']  # '1' = intercept, 'hours' = slope
}

result = pyinla(model=model, family='gaussian', data=df)

print("Bayesian Regression Results:")
print(result.summary_fixed)
print(f"\nTrue values: intercept={true_intercept}, hours={true_slope}")

In [ ]:
# Visualize the posterior distribution for the slope
marg_hours = result.marginals_fixed['hours']

plt.figure(figsize=(8, 4))
plt.fill_between(marg_hours['x'], marg_hours['y'], alpha=0.3, color='blue')
plt.plot(marg_hours['x'], marg_hours['y'], 'b-', linewidth=2, label='Posterior')
plt.axvline(true_slope, color='red', linestyle='--', linewidth=2, label=f'True value: {true_slope}')
plt.xlabel('Effect of Hours Studied (points per hour)')
plt.ylabel('Posterior Density')
plt.title('Posterior Distribution: Effect of Study Time on Test Score')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print("Interpretation: Each additional hour of studying increases the score by ~5 points.")

## 3. Poisson Regression: Event Counts

Now let's model count data - the number of customer arrivals at a store based on whether it's a weekend.

In [ ]:
# Simulate customer arrival data
np.random.seed(123)
n = 60  # 60 days of data

# 0 = weekday, 1 = weekend
is_weekend = np.random.binomial(1, 2/7, n)  # ~2/7 days are weekends

# True model: log(arrivals) = 3 + 0.5 * weekend
# Weekday: exp(3) ≈ 20 customers
# Weekend: exp(3.5) ≈ 33 customers
true_intercept = 3.0
true_weekend_effect = 0.5

log_rate = true_intercept + true_weekend_effect * is_weekend
arrivals = np.random.poisson(np.exp(log_rate))

df_counts = pd.DataFrame({
    'arrivals': arrivals,
    'weekend': is_weekend
})

print("Sample data:")
print(df_counts.head(10))
print(f"\nAverage arrivals on weekdays: {df_counts[df_counts['weekend']==0]['arrivals'].mean():.1f}")
print(f"Average arrivals on weekends: {df_counts[df_counts['weekend']==1]['arrivals'].mean():.1f}")

In [ ]:
# Fit Poisson regression
model_poisson = {
    'response': 'arrivals',
    'fixed': ['1', 'weekend']
}

result_poisson = pyinla(model=model_poisson, family='poisson', data=df_counts)

print("Poisson Regression Results (log scale):")
print(result_poisson.summary_fixed)
print(f"\nTrue values: intercept={true_intercept}, weekend={true_weekend_effect}")

# Interpret results
est_intercept = result_poisson.summary_fixed.loc['(Intercept)', 'mean']
est_weekend = result_poisson.summary_fixed.loc['weekend', 'mean']

print(f"\nInterpretation:")
print(f"  Expected weekday arrivals: exp({est_intercept:.2f}) = {np.exp(est_intercept):.1f}")
print(f"  Expected weekend arrivals: exp({est_intercept:.2f} + {est_weekend:.2f}) = {np.exp(est_intercept + est_weekend):.1f}")
print(f"  Weekend multiplier: exp({est_weekend:.2f}) = {np.exp(est_weekend):.2f}x more customers")

## 4. Random Effects: Students in Different Schools

When data has a grouped structure (students nested in schools), we can use random effects to account for school-level variation.

In [ ]:
# Simulate hierarchical data: students in schools
np.random.seed(456)

n_schools = 8
students_per_school = 15
n_total = n_schools * students_per_school

# School-level random effects (some schools are better than others)
school_effects = np.random.normal(0, 8, n_schools)

# Generate data
school_id = np.repeat(range(1, n_schools + 1), students_per_school)
hours = np.random.uniform(2, 8, n_total)

# True model: score = 50 + 4*hours + school_effect + noise
school_effect_expanded = np.array([school_effects[s-1] for s in school_id])
score = 50 + 4 * hours + school_effect_expanded + np.random.normal(0, 5, n_total)
score = np.clip(score, 0, 100)

df_schools = pd.DataFrame({
    'score': score,
    'hours': hours,
    'school': school_id
})

print("Sample data:")
print(df_schools.head(10))
print(f"\nSchools: {n_schools}, Students per school: {students_per_school}")

In [ ]:
# Visualize school differences
plt.figure(figsize=(10, 5))
for school in range(1, n_schools + 1):
    mask = df_schools['school'] == school
    plt.scatter(df_schools.loc[mask, 'hours'], df_schools.loc[mask, 'score'], 
                label=f'School {school}', alpha=0.7)
plt.xlabel('Hours Studied')
plt.ylabel('Test Score')
plt.title('Test Scores by School (notice different baselines)')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Fit mixed effects model
model_mixed = {
    'response': 'score',
    'fixed': ['1', 'hours'],
    'random': {
        'school': {'model': 'iid'}  # Random intercept per school
    }
}

result_mixed = pyinla(model=model_mixed, family='gaussian', data=df_schools)

print("Fixed Effects (population-level):")
print(result_mixed.summary_fixed)

print("\nHyperparameters (includes school variance):")
print(result_mixed.summary_hyperpar)

print("\nInterpretation:")
print("  - The 'hours' effect tells us how much each hour of study helps (on average)")
print("  - The school random effect captures baseline differences between schools")

## 5. Try Your Own Data!

You can upload your own CSV file and fit models to it.

In [ ]:
# === OPTION 1: Paste your data directly ===
from io import StringIO

# Replace this with your own data:
my_csv = """
x,y
1,3.2
2,5.1
3,6.8
4,9.3
5,11.0
6,12.5
7,15.1
8,16.8
"""

my_data = pd.read_csv(StringIO(my_csv))
print("Your data:")
print(my_data)

In [ ]:
# === OPTION 2: Upload a file ===
# Uncomment the code below to upload a CSV file:

# from google.colab import files
# uploaded = files.upload()
# filename = list(uploaded.keys())[0]
# my_data = pd.read_csv(filename)
# print(my_data.head())

In [ ]:
# Fit a model to your data
my_model = {
    'response': 'y',      # Change to your response variable name
    'fixed': ['1', 'x']   # Change to your predictor variable names
}

my_result = pyinla(model=my_model, family='gaussian', data=my_data)
print("Results for your data:")
print(my_result.summary_fixed)

## 6. What's Next?

This demo covered the basics. pyINLA supports much more:

**Likelihood families:**
- `gaussian` - Continuous data
- `poisson` - Count data
- `binomial` - Binary/proportion data
- `gamma`, `beta` - Positive/bounded continuous
- And 15+ more!

**Random effects:**
- `iid` - Independent random effects (shown above)
- `rw1`, `rw2` - Random walks for time series
- `ar1` - Autoregressive
- `besag`, `bym2` - Spatial areal models
- `spde` - Continuous spatial fields

### Learn More

- **Documentation**: [pyinla.org/docs](https://pyinla.org/docs)
- **Examples**: [pyinla.org/docs/examples](https://pyinla.org/docs/examples)
- **Applications**: [pyinla.org/apps](https://pyinla.org/apps)